# NLP Processing for Drug Intelligence

## Objective

The objective of this notebook is to extract meaningful insights from drug reviews using natural language processing techniques.

## Key Tasks

* Perform sentiment analysis on patient reviews
* Extract side effects from text data
* Generate drug-level insights

## Importance

This step transforms unstructured text into structured information that can be used for:

* effectiveness prediction
* decision making
* business insights

## Output

An enriched dataset containing:

* sentiment scores
* extracted side effects
* aggregated drug-level metrics


In [1]:
import pandas as pd

df = pd.read_csv("../data/clean_drug_reviews.csv")

df.head()

,drugName,condition,review,rating,date,usefulCount,clean_review,year,month,review_length,word_count,rating_category
0,Cyclosporine,keratoconjunctivitis sicca,"""i have used restasis for about a year now and...",2.0,2013-04-20,69,i have used restasis for about a year now and ...,2013,4,724,142,low
1,Etonogestrel,birth control,"""my experience has been somewhat mixed. i have...",7.0,2016-08-07,4,my experience has been somewhat mixed i have b...,2016,8,716,134,medium
2,Implanon,birth control,"""this is my second implanon would not recommen...",1.0,2016-05-11,6,this is my second implanon would not recommend...,2016,5,713,138,low
3,Hydroxyzine,anxiety,"""i recommend taking as prescribed, and the bot...",10.0,2012-03-19,124,i recommend taking as prescribed and the bottl...,2012,3,518,102,high
4,Dalfampridine,multiple sclerosis,"""i have been on ampyra for 5 days and have bee...",9.0,2010-08-01,101,i have been on ampyra for days and have been ...,2010,8,361,72,high


# Sentiment Analysis

Sentiment analysis is performed using a rule-based model (VADER), which is well-suited for short review text.

The output is a sentiment score and corresponding label.


In [3]:
!pip install nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 13.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [nltk]2/3 [nltk]]


In [4]:
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk

nltk.download('vader_lexicon')

sia = SentimentIntensityAnalyzer()

df['sentiment_score'] = df['clean_review'].apply(
    lambda x: sia.polarity_scores(x)['compound']
)

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/jatinchhabra/nltk_data...


In [5]:
def get_sentiment_label(score):
    if score > 0.05:
        return 'positive'
    elif score < -0.05:
        return 'negative'
    else:
        return 'neutral'

df['sentiment_label'] = df['sentiment_score'].apply(get_sentiment_label)

df[['clean_review', 'sentiment_score', 'sentiment_label']].head()

,clean_review,sentiment_score,sentiment_label
0,i have used restasis for about a year now and ...,-0.8748,negative
1,my experience has been somewhat mixed i have b...,0.9294,positive
2,this is my second implanon would not recommend...,-0.1139,negative
3,i recommend taking as prescribed and the bottl...,0.8148,positive
4,i have been on ampyra for days and have been ...,0.9204,positive


# Side Effect Extraction

Side effects are extracted from review text using a keyword-based approach.

This serves as a baseline method and can be extended using advanced NLP models such as Named Entity Recognition.


In [6]:
side_effect_keywords = [
    'nausea','headache','dizziness','fatigue','vomiting',
    'diarrhea','insomnia','anxiety','rash','pain',
    'fever','weight gain','weight loss','dry mouth',
    'blurred vision','constipation','sweating','weakness'
]

In [7]:
def extract_side_effects(text):
    effects = []
    for effect in side_effect_keywords:
        if effect in text:
            effects.append(effect)
    return effects

df['side_effects'] = df['clean_review'].apply(extract_side_effects)

In [8]:
df[['clean_review', 'side_effects']].head(10)

,clean_review,side_effects
0,i have used restasis for about a year now and ...,[]
1,my experience has been somewhat mixed i have b...,[]
2,this is my second implanon would not recommend...,[]
3,i recommend taking as prescribed and the bottl...,[]
4,i have been on ampyra for days and have been ...,[]
5,used for birth control and period issues very ...,[anxiety]
6,my prep instructions were one oz bottle the ev...,[]
7,love it i had the worst breakouts so i got epi...,[]
8,i felt a positive difference within the first ...,[anxiety]
9,i have been on mirena for over a year now and ...,"[headache, dizziness, insomnia]"


# Drug-Level Aggregation

We aggregate review-level insights into drug-level metrics to enable:

* comparison across drugs
* business decision-making
* dashboard visualization


In [9]:
drug_nlp_df = df.groupby('drugName').agg({
    'sentiment_score': 'mean',
    'rating': 'mean',
    'review': 'count',
    'usefulCount': 'mean'
}).reset_index()

drug_nlp_df.rename(columns={
    'sentiment_score': 'avg_sentiment',
    'rating': 'avg_rating',
    'review': 'review_count',
    'usefulCount': 'avg_usefulness'
}, inplace=True)

In [10]:
## TOP SIDE EFFECTS PER DRUG 

from collections import Counter

def get_top_effects(effects_list):
    all_effects = []
    for effects in effects_list:
        all_effects.extend(effects)
    return Counter(all_effects).most_common(3)

side_effects_per_drug = df.groupby('drugName')['side_effects'].apply(list)

drug_nlp_df['top_side_effects'] = drug_nlp_df['drugName'].map(
    side_effects_per_drug.apply(get_top_effects)
)

In [11]:
drug_nlp_df['sentiment_rating_interaction'] = (
    drug_nlp_df['avg_sentiment'] * drug_nlp_df['avg_rating']
)

In [12]:
df.to_csv("../data/nlp_processed_reviews.csv", index=False)
drug_nlp_df.to_csv("../data/drug_nlp_summary.csv", index=False)